In [1]:
import sys
sys.path.append('../../Simulate/')

from StreamMethDB import StreamMethDB
from SetMethylation import SetMethylation
from UtilityFunctions import parseCGmap, parseASM
from UtilityFunctions import retrieve_iupac

In [2]:
import os
import re
import warnings
import pandas as pd
import numpy as np
import subprocess
import threading

from Bio import SeqIO
from scipy.stats import beta
from tqdm import tqdm
from typing import Dict, List

In [3]:
class StreamWGSIM:
    '''
    stream WGSIM output for bisulfite reads generation
    :param str  sim_cmd : WGSIM commands for simulation
    :param bool pair_end: pair_end or not
    :rtype None
    '''
    def __init__(self, sim_cmd: list = None, pair_end: bool = True):
        self.sim_cmd  = sim_cmd
        self.pair_end = pair_end

    def __iter__(self):
        wgsim = subprocess.Popen(self.sim_cmd, stdout=subprocess.PIPE, universal_newlines=True)
        sim_iter = iter(wgsim.stdout.readline, b'')

        line  = self.get_line(sim_iter) # line is None when EOF
        while line:
            # collect all variant lines on the contig, after that sim_iter points to read lines
            if line == "Contig Variant Start":
                variant_contig, variant_dict = self.collect_variants(sim_iter)
                yield variant_contig, variant_dict

            # collect read pairs
            for collect_flag, read_pair in self.collect_reads(sim_iter):
                if collect_flag: # {1: collect_reads, 0: swith to collect_vars or EOF}
                    yield False, read_pair
                else:
                    line = "Contig Variant Start" if isinstance(read_pair, list) else None
                    break

    def collect_variants(self, sim_iter):
        '''collect variant lines from stdout'''
        variant_dict = {}
        variant_info = {}

        while True:
            line = self.get_line(sim_iter)
            if line == 'Contig Variant End':
                return variant_info['chrom'], variant_dict

            variant_info = self.process_variant_line(line)
            if variant_info['pos']:
                assert variant_info['pos'] not in variant_dict
                variant_dict[variant_info['pos']] = variant_info


    def collect_reads(self, sim_iter):
        '''collect read lines from stdout'''
        skip_flag = not self.pair_end

        while True:
            line  = self.get_line(sim_iter)
            if not line: # EOF
                yield 0, None
            elif line == "Contig Variant Start": # switch to collect variants
                yield 0, []
            else:
                read1 = self.process_read_lines(sim_iter, line = line)
                read2 = self.process_read_lines(sim_iter, skip = skip_flag)
                yield 1, [read1, read2]


    @staticmethod
    def get_line(sim_iter):
        '''receive lines from console'''
        try:
            line = next(sim_iter).strip()
        except StopIteration:
            print("End of output\n")
            return None
        else:
            return line


    @staticmethod
    def process_variant_line(line: str) -> Dict:
        '''parse variant lines'''
        line_split = line.split('\t')

        try:
            chrom, pos, ref, alt, heter_flag = line_split
        except ValueError:
            return dict(chrom=line_split[0], pos = None)
        else:
            heter = heter_flag == '+'
            indel = int(ref == '-') - int(alt == '-') # 1 for ref=='-', -1 for alt=='-', o.w. 0
            offset= indel * max(len(ref), len(alt))
            if indel:
                iupac  = None
            else:
                iupac  = retrieve_iupac(alt)
                alt    = list(set(iupac) - set(ref))[0]
            return dict(chrom=chrom, pos=int(pos), ref=ref, alt=alt,
                        offset=offset, heter=heter, indel=indel, iupac=iupac)


    @staticmethod
    def process_read_lines(sim_iter, line = None, skip = False):
        '''parse read lines'''
        if skip:
            next(sim_iter)
            next(sim_iter)
            next(sim_iter)
            next(sim_iter)
            return None

        if not line:
            line = next(sim_iter).strip()
        # header, seq, comment process
        read_id, pair, flag_pos, flag_mut, flag_indel, qual, cgr = line.split(' ')
        cgr = np.frombuffer(cgr.encode(), dtype=np.int8)
        seq = np.frombuffer(next(sim_iter).strip().encode(), dtype=np.int8)
        _, start, end, cover_pos, n_sub, n_indel, insert_size, inner_dist, ofs= next(sim_iter).strip().split(':')
        ofs = np.fromstring(ofs, dtype=np.int8, sep = ',')
        ctx = np.frombuffer(next(sim_iter).strip().encode(), np.int8)
        return dict(read_id=read_id, pair=int(pair), qual = int(qual),
                    flag_pos=int(flag_pos), flag_mut=int(flag_mut), flag_indel=int(flag_indel),
                    start=int(start), end=int(end), cover_pos=int(cover_pos),
                    n_sub=int(n_sub), n_indel=int(n_indel),
                    insert_size=int(insert_size), inner_dist=int(inner_dist),
                    cgr=cgr, seq=seq, ofs=ofs, ctx=ctx)


In [4]:
class LockedIterator(object):
    '''make generator/iterator thread safe'''
    def __init__(self, it):
        self.lock = threading.Lock()
        self.it = iter(it)

    def __iter__(self): 
        return self

    def __next__(self):
        with self.lock:
            return self.it.__next__()

In [5]:
ref_fasta = "/home/wbguo/iproject/BSReadSim/test/ref/BSB_test.fa"
cgmap_file= "/home/wbguo/iproject/BSReadSim/temp/data/sim.CGmap.gz"
asm_file  = "/home/wbguo/iproject/BSReadSim/temp/data/sim.asm.gz"

In [6]:
# meth_set = SetMethylation(ref_fasta=ref_fasta, cgmap_file=cgmap_file, asm_file = asm_file,
#                           outdir="/home/wbguo/iproject/BSReadSim/temp/outdir/",
#                           overwrite_db=True, verbose = True)

In [7]:
meth_set = SetMethylation(meth_db_path="/home/wbguo/iproject/BSReadSim/temp/outdir/",
                          outdir="/home/wbguo/iproject/BSReadSim/temp/outdir/",
                          ref_fasta = ref_fasta, overwrite_db=True)

In [8]:
sim_cmd_part = ['/home/wbguo/iproject/BSReadSim/WGSIM/wgsim', 
                '-1', '100', '-2', '100','-e','0.005','-d','400','-s','25',
                '-r','0.1', '-N','1000',
                '-R','0.15','-X','0.15',
                '-S','2022',
                '-A','0.05','-h','0', '-m', '1', ref_fasta]

In [9]:
for contig_id in meth_set.ref_dict.keys():
    sim_cmd  = sim_cmd_part + ['-c', contig_id]
    read_gen = LockedIterator(StreamWGSIM(sim_cmd=sim_cmd))
    var_contig, sim_data= next(read_gen)
    break

[wgsim] seed = 2022
[wgsim_core] calculating the total length of the reference sequence...
[wgsim_core] 6 contig sequences, total length: 1961600
[wgsim_core] Simulate 216 reads from contig chr10 (calculate from -N, as -n is not specified)...
[wgsim_core] No VCF input, will generate SNP randomly if mutation rate is nonzero


In [10]:
contig_id = list(meth_set.ref_dict.keys())[0]

In [11]:
sim_cmd  = sim_cmd_part + ['-c', contig_id]
read_gen = LockedIterator(StreamWGSIM(sim_cmd=sim_cmd))
var_contig, sim_data= next(read_gen)

[wgsim] seed = 2022
[wgsim_core] calculating the total length of the reference sequence...
[wgsim_core] 6 contig sequences, total length: 1961600
[wgsim_core] Simulate 216 reads from contig chr10 (calculate from -N, as -n is not specified)...
[wgsim_core] No VCF input, will generate SNP randomly if mutation rate is nonzero


In [12]:
var_contig

'chr10'

In [13]:
sim_data

{28: {'chrom': 'chr10',
  'pos': 28,
  'ref': 'T',
  'alt': 'A',
  'offset': 0,
  'heter': True,
  'indel': 0,
  'iupac': ('A', 'T')},
 35: {'chrom': 'chr10',
  'pos': 35,
  'ref': 'T',
  'alt': 'G',
  'offset': 0,
  'heter': True,
  'indel': 0,
  'iupac': ('G', 'T')},
 45: {'chrom': 'chr10',
  'pos': 45,
  'ref': 'C',
  'alt': 'T',
  'offset': 0,
  'heter': True,
  'indel': 0,
  'iupac': ('C', 'T')},
 47: {'chrom': 'chr10',
  'pos': 47,
  'ref': 'T',
  'alt': 'G',
  'offset': 0,
  'heter': False,
  'indel': 0,
  'iupac': ('G',)},
 91: {'chrom': 'chr10',
  'pos': 91,
  'ref': 'G',
  'alt': 'A',
  'offset': 0,
  'heter': False,
  'indel': 0,
  'iupac': ('A',)},
 93: {'chrom': 'chr10',
  'pos': 93,
  'ref': 'T',
  'alt': 'C',
  'offset': 0,
  'heter': True,
  'indel': 0,
  'iupac': ('C', 'T')},
 108: {'chrom': 'chr10',
  'pos': 108,
  'ref': 'C',
  'alt': 'G',
  'offset': 0,
  'heter': True,
  'indel': 0,
  'iupac': ('G', 'C')},
 112: {'chrom': 'chr10',
  'pos': 112,
  'ref': 'A',
  'alt

In [14]:
pos_map, meth_arr, _ = meth_set.meth_db.load_contig(contig_id)

In [15]:
y = meth_set.set_var_meth(var_contig, sim_data)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 42318/42318 [00:07<00:00, 5542.37it/s]


In [16]:
y

{35: (0.0, 15),
 47: (0.0, 15),
 93: (0.000838, 1),
 108: (0.432, 9),
 125: (6e-08, 11),
 170: (array([1.], dtype=float16), array([3], dtype=int16)),
 181: (0.0, 7),
 235: (array([0.], dtype=float16), array([7], dtype=int16)),
 264: (array([0.], dtype=float16), array([7], dtype=int16)),
 295: (0.0, 15),
 297: (0.01807, 7),
 314: (0.0, 3),
 378: (0.0, 7),
 383: (array([0.8047, 0.    ], dtype=float16), array([9, 0], dtype=int16)),
 405: (0.0, 15),
 407: (0.881, 15),
 429: (0.0, 15),
 430: (0.9995, 1),
 443: (0.969, 9),
 451: (0.0, 15),
 460: (0.0, 7),
 492: (0.0, 11),
 498: (0.751, 1),
 523: (1.0, 3),
 564: (0.001792, 15),
 566: (0.0, 15),
 572: (0.997, 15),
 647: (0.00055, 15),
 688: (1.0, 7),
 691: (6e-08, 15),
 703: (0.0, 11),
 739: (0.00973, 11),
 755: (0.0, 15),
 772: (0.787, 9),
 792: (1.81e-05, 7),
 826: (0.0, 15),
 831: (9.5e-07, 11),
 865: (0.0525, 7),
 881: (array([1.], dtype=float16), array([7], dtype=int16)),
 892: (0.978, 1),
 905: (1.0, 7),
 958: (0.0, 11),
 968: (0.0, 11),

In [17]:
from pympler import asizeof

In [18]:
asizeof.asizeof(y)

3325696

In [19]:
len(y)

20126